# Physics-Informed Neural Networks
## Solving a differential equation without solving it

In the previous notebook we stored a field we already knew, as a network taking
coordinates. Here the field is **unknown**, and we are going to find it &mdash; not by
fitting the network to values, but by fitting it to the **equation**.

The example is the damped harmonic oscillator:
\begin{equation}
    m\ddot{u} + c\dot{u} + k u = 0, \qquad u(0) = 1, \qquad \dot{u}(0) = 0
\end{equation}
which we write with $m = 1$, $c = \mu$ and a stiffness $k$. It has a closed-form
solution, which we will only use to check our results.

The situation we simulate: a sensor that failed early. We have a few noisy
measurements over the first third of the interval, and nothing afterwards.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

MU, K = 2.0, 100.0        # damping and stiffness
T = 1.5                   # time horizon
T_DATA = 0.5              # the sensor stops here

ZETA = MU / (2 * np.sqrt(K))
W0 = np.sqrt(K)
WD = W0 * np.sqrt(1 - ZETA ** 2)


def exact(t):
    """Analytical solution, used for checking only."""
    t = np.asarray(t, dtype=float)
    return np.exp(-ZETA * W0 * t) * (np.cos(WD * t)
                                     + (ZETA * W0 / WD) * np.sin(WD * t))


rng = np.random.default_rng(3)
t_data = np.linspace(0.0, T_DATA, 12)
u_data = exact(t_data) + rng.normal(0, 0.015, t_data.shape)

t_plot = np.linspace(0, T, 900)

plt.figure(figsize=(9, 3.4))
plt.axvspan(T_DATA, T, color="0.94", zorder=0)
plt.plot(t_plot, exact(t_plot), color="C0", lw=2, label="exact solution", zorder=2)
plt.scatter(t_data, u_data, color="C1", zorder=3, label="observations")
plt.text((T_DATA + T) / 2, 1.1, "no data at all", ha="center", style="italic",
         color="0.45")
plt.xlabel("t"); plt.ylabel("u(t)"); plt.legend(loc="lower right"); plt.show()

In [ ]:
def make_net(width=64, depth=3, seed=0):
    """A small coordinate network:  t -> u(t)."""
    keras.utils.set_random_seed(seed)
    layers = [keras.Input(shape=(1,))]
    for _ in range(depth):
        layers.append(keras.layers.Dense(width, activation="tanh"))
    layers.append(keras.layers.Dense(1))
    return keras.Sequential(layers)

## 1. The baseline: ordinary supervised learning

Before adding any physics, let us do what lectures 3 and 4 taught us: fit a network
to the measurements, with `compile` and `fit`.

<span style="color:blue">Train a network on the 12 observations, then predict over the whole
interval $[0, T]$ and plot the result against the exact solution. Use the Adam
optimizer with a learning rate of <code>2e-3</code> and a mean squared error loss,
for 3000 epochs (full batch).</span>

In [ ]:
net_data = make_net()
# TODO: compile the network
...
# TODO: fit it on (t_data, u_data) for 3000 epochs, full batch, verbose=0
...

u_pred = net_data.predict(t_plot.reshape(-1, 1), verbose=0).ravel()

plt.figure(figsize=(9, 3.4))
plt.axvspan(T_DATA, T, color="0.94", zorder=0)
plt.plot(t_plot, exact(t_plot), color="C0", lw=2, label="exact", zorder=2)
plt.plot(t_plot, u_pred, "--", color="C1", lw=2, label="network", zorder=3)
plt.scatter(t_data, u_data, color="C1", zorder=4)
plt.xlabel("t"); plt.ylabel("u(t)"); plt.legend(); plt.show()

<span style="color:blue">The network fits the observations almost perfectly. Is it a good model
of the oscillator? What exactly went wrong, and could <i>any</i> amount of training
have fixed it?</span>

## 2. Differentiating the network with respect to its input

The network is a differentiable function of $t$. TensorFlow can give us
$\dot{u}_\theta$ and $\ddot{u}_\theta$ **exactly**, by recording the operations on a
`GradientTape` &mdash; the same mechanism as the backpropagation of lecture 3, except
that we differentiate with respect to the **input** instead of the weights.

A second derivative needs two nested tapes: the outer one records the computation of
the first derivative.

<span style="color:blue">Complete the two missing gradient calls below.</span>

In [ ]:
def derivatives(net, t):
    """Return u, du/dt and d2u/dt2 at the times t (a (n, 1) tf tensor)."""
    with tf.GradientTape() as tape2:
        tape2.watch(t)
        with tf.GradientTape() as tape1:
            tape1.watch(t)
            u = net(t)
        # TODO: first derivative of u with respect to t
        u_t = ...
    # TODO: derivative of u_t with respect to t
    u_tt = ...
    return u, u_t, u_tt

Let us check it on the network we just trained, by comparing the
derivative given by the tape with a finite-difference estimate of the same quantity.

In [ ]:
tc = tf.constant(t_plot.reshape(-1, 1), dtype=tf.float32)
u, u_t, u_tt = derivatives(net_data, tc)

plt.figure(figsize=(9, 3.4))
plt.plot(t_plot, u_t.numpy().ravel(), color="C0", lw=2, label="autodiff")
plt.plot(t_plot, np.gradient(u.numpy().ravel(), t_plot), "--", color="C1", lw=2,
         label="finite differences on the same curve")
plt.xlabel("t"); plt.ylabel("du/dt"); plt.legend(); plt.show()

## 3. Fitting the equation instead of the data

We now have everything we need to **evaluate the equation** at any time $t$, without
knowing its solution. The residual
\begin{equation}
    r_\theta(t) = \ddot{u}_\theta(t) + \mu \dot{u}_\theta(t) + k\, u_\theta(t)
\end{equation}
is zero exactly when $u_\theta$ solves the problem. Minimising $r_\theta^2$ over a
cloud of **collocation times** therefore pushes the network towards the solution.

The total loss has three terms:
\begin{equation}
    \mathcal{L}(\theta) = \lambda_d \mathcal{L}_{\text{data}}
                         + \mathcal{L}_{\text{residual}}
                         + \lambda_{ic} \mathcal{L}_{\text{init}}
\end{equation}

Note that this loss is **not** a function of a predicted value and a target value, so
it cannot be passed to `compile`. We have to write the training loop ourselves.

<span style="color:blue">Complete the residual and the total loss in the training loop below.
Scale the residual by $k$ so that the three terms have comparable magnitudes.</span>

In [ ]:
def train_pinn(iters=16000, n_col=800, lam_data=10.0, lam_ic=10.0,
               lr=2e-3, seed=0):
    net = make_net(seed=seed)
    td = tf.constant(t_data.reshape(-1, 1), dtype=tf.float32)
    ud = tf.constant(u_data.reshape(-1, 1), dtype=tf.float32)
    tcol = tf.constant(np.linspace(0, T, n_col).reshape(-1, 1), dtype=tf.float32)
    t0 = tf.constant([[0.0]], dtype=tf.float32)

    sched = keras.optimizers.schedules.ExponentialDecay(lr, iters // 3, 0.4,
                                                        staircase=True)
    opt = keras.optimizers.Adam(learning_rate=sched)

    @tf.function
    def step():
        with tf.GradientTape() as tape:
            loss_data = tf.reduce_mean((net(td) - ud) ** 2)

            u, u_t, u_tt = derivatives(net, tcol)
            # TODO: the residual of  u'' + MU u' + K u = 0,  divided by K
            res = ...
            loss_res = tf.reduce_mean(res ** 2)

            _, u0_t, _ = derivatives(net, t0)
            loss_ic = tf.reduce_mean((net(t0) - 1.0) ** 2 + (u0_t / W0) ** 2)

            # TODO: combine the three terms with their weights
            loss = ...
        grads = tape.gradient(loss, net.trainable_variables)
        opt.apply_gradients(zip(grads, net.trainable_variables))
        return loss

    for i in range(iters):
        loss = step()
        if i % 4000 == 0:
            print(f"  iter {i:6d}   loss {float(loss):.3e}")
    return net

Training takes a minute or two: each iteration runs three backward
passes instead of one.

In [ ]:
net_pinn = train_pinn()
u_pinn = net_pinn.predict(t_plot.reshape(-1, 1), verbose=0).ravel()

print("max error, data only :", np.abs(u_pred - exact(t_plot)).max().round(3))
print("max error, PINN      :", np.abs(u_pinn - exact(t_plot)).max().round(3))

fig, axes = plt.subplots(2, 1, figsize=(9, 6.4), sharex=True)
for ax, u, ttl in zip(axes, [u_pred, u_pinn],
                      ["data only", "data + PDE residual (PINN)"]):
    ax.axvspan(0, T_DATA, color="0.94", zorder=0)
    ax.plot(t_plot, exact(t_plot), color="C0", lw=2, label="exact", zorder=2)
    ax.plot(t_plot, u, "--", color="C1", lw=2, label="network", zorder=3)
    ax.scatter(t_data, u_data, color="C1", s=25, zorder=4)
    ax.set_title(ttl, loc="left"); ax.set_ylabel("u(t)"); ax.set_ylim(-1.2, 1.8)
axes[1].set_xlabel("t"); axes[0].legend(loc="upper right")
plt.show()

<span style="color:blue">Set <code>lam_ic = 0.0</code> and train again. The equation is still
enforced everywhere &mdash; so why does the result change? What does this tell you
about what the residual term alone can and cannot determine?</span>

## 4. Bonus: the inverse problem

So far the stiffness $k$ was known. Suppose it is not: we have measurements of an
oscillator over the whole interval, and we want to **identify** $k$ from them.

With a classical approach this means an outer optimization loop, each iteration of
which runs a full solve. With a PINN, $k$ is simply one more trainable variable,
optimized jointly with the weights &mdash; at no extra cost.

<span style="color:blue">The unknown stiffness is declared below as a <code>tf.Variable</code>,
with a deliberately wrong initial guess. Complete the residual so that it uses the
<i>estimated</i> stiffness, and add the variable to the list of parameters the
optimizer updates.</span>

In [ ]:
t_obs = np.linspace(0, T, 40)
u_obs = exact(t_obs) + rng.normal(0, 0.02, t_obs.shape)

net_inv = make_net(seed=1)
# the unknown, parameterized through its logarithm so that it stays positive
log_k = tf.Variable(np.log(40.0), dtype=tf.float32)

td = tf.constant(t_obs.reshape(-1, 1), dtype=tf.float32)
ud = tf.constant(u_obs.reshape(-1, 1), dtype=tf.float32)
tcol = tf.constant(np.linspace(0, T, 800).reshape(-1, 1), dtype=tf.float32)
opt = keras.optimizers.Adam(1e-3)


@tf.function
def step_inv():
    with tf.GradientTape() as tape:
        k_hat = tf.exp(log_k)
        loss_data = tf.reduce_mean((net_inv(td) - ud) ** 2)
        u, u_t, u_tt = derivatives(net_inv, tcol)
        # TODO: residual using the estimated stiffness k_hat
        res = ...
        loss = 10.0 * loss_data + tf.reduce_mean(res ** 2)
    # TODO: the variables to update are the network weights AND log_k
    variables = ...
    opt.apply_gradients(zip(tape.gradient(loss, variables), variables))
    return loss


for i in range(12000):
    loss = step_inv()
    if i % 3000 == 0:
        print(f"  iter {i:5d}   loss {float(loss):.3e}   k = {float(tf.exp(log_k)):.1f}")

print(f"\nrecovered k = {float(tf.exp(log_k)):.2f}   (true value: {K})")

<span style="color:blue">How many times was the differential equation <i>solved</i> during this
identification? Compare with what a classical least-squares identification would have
required.</span>

## What to remember

* The network is the **solution**, not a predictor: its input is a coordinate, its
  output the value of the field there.
* Training no longer means learning from data &mdash; it means **solving**. The
  residual term needs no labels at all.
* This only works because the derivatives with respect to the inputs are available
  exactly, through automatic differentiation.
* The weights $\lambda$ of the loss terms are real hyperparameters: the same equation
  with different weights gives different answers.
* Changing the initial condition, or any coefficient, means **retraining from
  scratch**. One network solves one problem instance &mdash; which is exactly what
  neural operators will address.